# Ejercicio 6, Parte 4: análisis de las simulaciones

Lee los CSV que genera BehaviorSpace con `ants_original_sim.nlogox` y `ants_levy_sim.nlogox`
y los deja en una sola tabla de pandas, **una fila por corrida**:

| columna | qué es |
|---|---|
| `modelo` | `"original"` o `"levy"` |
| `run` | número de corrida de BehaviorSpace |
| `population`, `diffusion-rate`, `evaporation-rate`, `eta` | parámetros de la corrida (`eta` es NaN en el original) |
| `steps` | ticks que duró la corrida |
| `descubre-1`, `descubre-2`, `descubre-3` | tick en que se encontró cada montón (−1 = nunca) |
| `todos-descubiertos` | tick en que ya estaban los tres |
| `tick-fin` | tick en que se terminó de llevar la comida al nido (−1 = no terminó antes del límite) |

Sirve tanto para la salida *Spreadsheet* como para la *Table* de BehaviorSpace.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
DATOS = Path("datos")
FIGURAS = Path("../../figuras")
METRICAS = ["descubre-1", "descubre-2", "descubre-3", "todos-descubiertos", "tick-fin"]

## Lectura de los CSV

BehaviorSpace guarda dos formatos:

- **Table:** 6 renglones de encabezado y luego un renglón por corrida (y por tick, si se mide en cada paso). Se lee directo con `pd.read_csv`.
- **Spreadsheet:** cada corrida es un bloque de columnas (`[step]` más una por métrica). Los parámetros van en renglones arriba y el valor final de cada métrica en el renglón después de `[final value]`.

In [ ]:
def leer_spreadsheet(filas):
    etiquetas = [f[0] if f else "" for f in filas]
    i_run = etiquetas.index("[run number]")
    i_steps = etiquetas.index("[total steps]")
    i_final = etiquetas.index("[final value]")

    corridas = filas[i_run][1:]
    ancho = corridas.count(corridas[0])        # columnas por corrida: [step] + métricas
    nombres = filas[i_final][1:1 + ancho]
    valores = filas[i_final + 1][1:]

    registros = []
    for j in range(0, len(corridas), ancho):
        r = {"run": int(corridas[j]), "steps": int(filas[i_steps][1 + j])}
        for fila in filas[i_run + 1:i_steps]:   # renglones de parámetros
            r[fila[0]] = fila[1 + j]
        for k, nombre in enumerate(nombres):
            if nombre != "[step]":
                r[nombre] = valores[j + k]
        registros.append(r)
    return pd.DataFrame(registros)


def leer_table(ruta):
    df = pd.read_csv(ruta, skiprows=6)
    df = df.rename(columns={"[run number]": "run", "[step]": "steps"})
    # Si se midió en cada paso, quedarse con el último renglón de cada corrida
    return df.sort_values(["run", "steps"]).groupby("run", as_index=False).last()


def leer_behaviorspace(ruta):
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    if "Spreadsheet" in " ".join(filas[0]):
        df = leer_spreadsheet(filas)
    else:
        df = leer_table(ruta)
    # Todo llega como texto; convertir a número las columnas que lo sean
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass
    return df

In [ ]:
tablas = []
for modelo, patron in [("original", "ants_original_sim*.csv"), ("levy", "ants_levy_sim*.csv")]:
    for ruta in sorted(DATOS.glob(patron)):
        df = leer_behaviorspace(ruta)
        df.insert(0, "modelo", modelo)
        tablas.append(df)
        print(f"{ruta.name}: {len(df)} corridas")

datos = pd.concat(tablas, ignore_index=True)
if "eta" not in datos:
    datos["eta"] = np.nan
datos

## Revisión rápida

Corridas que no terminaron antes del límite de ticks (`tick-fin = -1`) o montones que nunca se encontraron.
Si hay, conviene quitarlas del promedio de esa métrica y reportarlas aparte.

In [ ]:
incompletas = datos[(datos[METRICAS] < 0).any(axis=1)]
print(f"{len(incompletas)} corridas incompletas de {len(datos)}")
incompletas

## Media y desviación estándar

Una fila por modelo (y por valor de `eta` en Lévy). Los −1 se cambian por NaN para que no entren al promedio.

In [ ]:
limpios = datos.copy()
limpios[METRICAS] = limpios[METRICAS].where(limpios[METRICAS] >= 0)

# dropna=False para que el original (eta = NaN) no desaparezca del groupby
resumen = (limpios.groupby(["modelo", "eta"], dropna=False)[METRICAS]
                  .agg(["mean", "std", "count"])
                  .round(1))
resumen

## Gráfica de ejemplo

Caja de cada métrica por configuración. Cada caja va de un cuartil al otro, la línea es la mediana y los puntos son valores atípicos.

In [ ]:
limpios["configuracion"] = np.where(
    limpios["modelo"] == "original", "original",
    "Lévy $\\mu=" + limpios["eta"].map("{:g}".format) + "$")
orden = list(dict.fromkeys(limpios.sort_values(["modelo", "eta"], ascending=[False, True])["configuracion"]))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, metrica, titulo in [(axs[0], "todos-descubiertos", "Los tres montones descubiertos"),
                            (axs[1], "tick-fin", "Toda la comida en el nido")]:
    grupos = [limpios.loc[limpios["configuracion"] == c, metrica].dropna() for c in orden]
    ax.boxplot(grupos, tick_labels=orden)
    ax.set_title(titulo)
    ax.set_ylabel("ticks")
    ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(FIGURAS / "comparacion_busqueda.pdf")
plt.show()

## Histogramas del tick de descubrimiento de cada montón

Un panel por montón y una curva por configuración. Se grafica la **fracción** de corridas en cada intervalo
(no el conteo) porque el original tiene 600 corridas y cada $\mu$ 200; así las curvas se pueden comparar.
Cada panel muestra hasta el percentil 99 de su montón, así que el 1 % de corridas más tardadas queda fuera de la vista (casi todas son de $\mu=3$); las medias sí las incluyen.

Usa `limpios`, `orden` y la columna `configuracion` de la celda anterior.

In [ ]:
MONTONES = [("descubre-1", "Montón 1 (el más cercano)"),
            ("descubre-2", "Montón 2"),
            ("descubre-3", "Montón 3 (el más lejano)")]
ANCHO_BIN = 5    # ticks por intervalo
COLORES = {c: f"C{i}" for i, c in enumerate(orden)}

fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (metrica, titulo) in zip(axs, MONTONES):
    # Cada panel con su propio rango: hasta el percentil 99 de ese montón, para que
    # las pocas corridas muy tardadas (casi todas de mu=3) no aplasten el resto
    limite = limpios[metrica].quantile(0.99)
    bins = np.arange(0, limpios[metrica].max() + ANCHO_BIN, ANCHO_BIN)
    for c in orden:
        valores = limpios.loc[limpios["configuracion"] == c, metrica].dropna()
        # weights = 1/n: cada barra es la fracción de corridas de esa configuración
        ax.hist(valores, bins=bins, weights=np.full(len(valores), 1 / len(valores)),
                histtype="step", lw=2, color=COLORES[c], label=c)
        ax.axvline(valores.mean(), color=COLORES[c], ls=":", lw=1.2)
    ax.set_xlim(0, limite)
    ax.set_title(titulo)
    ax.set_xlabel("tick de descubrimiento")
    ax.grid(alpha=0.3)
axs[0].set_ylabel("fracción de corridas")
axs[0].legend(title="línea punteada: media")
fig.tight_layout()
fig.savefig(FIGURAS / "histograma_descubrimientos.pdf")
plt.show()

## Tu espacio

`datos` tiene los valores crudos (con −1) y `limpios` los tiene como NaN. Algunas ideas:

```python
limpios.groupby(["modelo", "eta"], dropna=False)["descubre-3"].median()
limpios[limpios["modelo"] == "levy"].plot.scatter(x="eta", y="tick-fin")
```